# Ordered Logistic Regression Adoption Predictors Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset on adoption predictors in rangeland management using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Instantiate dataset from Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access metadata (as object)
meta = dataset.metadata
print(f"Dataset title: {meta.name}")
print(f"\nDescription: {meta.description}")

## 2. Data Overview
List available record sets defined in the dataset, display their IDs (`@id`), descriptive names, and fields. All are referenced by their `@id`.

In [ ]:
# List available record sets by @id and display their field @ids
print("Available record sets:")
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"  - RecordSet @id: {record_set.id}")
    print(f"    Name: {getattr(record_set, 'name', '[no name]')}")
    field_ids = [f.id for f in getattr(record_set, "fields", [])]
    print(f"    Fields (@id): {field_ids}")
    record_set_ids.append(record_set.id)

# Also print total number of record sets found
print(f"\nTotal record sets found: {len(record_set_ids)}")

## 3. Data Extraction
Load records from a selected record set using its `@id` into a pandas DataFrame for further analysis.

*Select the first record set for demonstration, and reference all fields by their `@id`.*

In [ ]:
# Use the discovered record set @ids from previous cell
dataframes = {}

for recset_id in record_set_ids:
    print(f"\nLoading records for RecordSet: {recset_id}")
    try:
        rows = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(rows)
        dataframes[recset_id] = df
        print(f"  => Loaded {len(df)} records. Columns (all by field @id): {list(df.columns)}")
    except Exception as err:
        print(f"  [Warning] Could not load records for {recset_id}: {err}")

# For further exploration, pick the first successfully loaded record set
main_recordset_id = None
for recset_id, df in dataframes.items():
    if not df.empty:
        main_recordset_id = recset_id
        break

if main_recordset_id:
    print(f"\nUsing record set: {main_recordset_id}")
    print(f"Sample data:")
    display(dataframes[main_recordset_id].head())
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)
Let's process the data from the selected record set. Typical steps include filtering records, normalizing numeric fields, and grouping by categorical fields.

*Fields and columns are referenced by their `@id` values.*

In [ ]:
# Pick an example numeric field for analysis (update the field_id based on your data)
if main_recordset_id:
    df = dataframes[main_recordset_id].copy()
    # Try to automatically find a numeric field
    numeric_field_id = None
    for col in df.columns:
        # Try to detect float/int type
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Analyzing numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile): {len(filtered_df)} rows")

        # Normalize and show example
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"First 5 rows of normalized data:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to pick a group field (categorical) for grouping
        group_field_id = None
        for col in df.columns:
            if df[col].nunique() < len(df[col]) / 2 and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Mean of the numeric field by group:")
            display(grouped_df.head())
        else:
            print("No suitable group (categorical) field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No loaded data to explore.")

## 5. Visualization
Visualize distributions or relationships in the dataset using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if a DataFrame and a numeric field are found
if main_recordset_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        # Box plot by group
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We reviewed record sets and fields (referenced by `@id`), extracted records as DataFrames, performed basic preprocessing, and visualized key distributions. For further analysis, consult the Croissant schema and field definitions. This approach helps ensure reproducibility and clarity by referencing data via their unique `@id` throughout your workflow.